# Perform spatial merge and intersection
- Big Q: How do driver and neighborhood characteristics factor into stop length?
- 2017 to 2023 timeframe

### Imports

In [78]:
import geopandas as gpd
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os
import numpy as np

### Spatial merge 

In [2]:
# Load reporting districts and census blocks shapefiles
reporting_districts = gpd.read_file('data/Law_Enforcement_Reporting_Districts')
# Filter the GeoDataFrame
reporting_districts = reporting_districts[reporting_districts['AGENCY'] == "LAPD"]

census_tracts = gpd.read_file('data/2020_Census_Tracts')

In [3]:
# Ensure both GeoDataFrames use the same CRS (needs to be projected, UTM zone 11N for LA)
census_tracts = census_tracts.to_crs(epsg=32611)
reporting_districts = reporting_districts.to_crs(epsg=32611)

In [4]:
# Perform the intersection
intersected = gpd.overlay(census_tracts, reporting_districts, how='intersection')

# Calculate the area of each intersected geometry (using sq feet because that's what is reported in raw data) and add it as a new column
intersected['Intersecting_Area_SqF'] = (intersected.geometry.area * 10.7639).astype(int)  # Conversion factor from square meters to square feet


In [5]:
# Rename columns using a dictionary
rename_dict = {
    'CT20': 'Census_Tract_ID',
    'ShapeSTAre': 'Census_Tract_Area',
    'ShapeSTLen': 'Census_Tract_Length',
    'RD':'Reporting_District_ID',
    'OMEGA_NAME': 'Reporting_District_Name',
    'Shape_Leng': 'Reporting_District_Length',
    'Shape_Area': 'Reporting_District_Area'
}

# Rename columns
intersected = intersected.rename(columns=rename_dict)
intersected = intersected.drop(columns={'OBJECTID_1','LABEL_1','OBJECTID_2','S_TYPE','COMMTYPE','NAME','ST_NAME','OMEGA_LABE','LABEL_2','AGENCY','STATION'})

### Plot and export CSV

In [6]:
# # Create plot
# ax = intersected.plot(figsize=(100, 100), alpha=0.5, label='Intersected')
# census_tracts.plot(ax=ax, facecolor='none', edgecolor='black', label='Census Tracts')
# reporting_districts.plot(ax=ax, facecolor='none', edgecolor='red', label='Reporting Districts')

# # Create legend handles and labels
# intersected_legend = plt.Rectangle((0,0), 1, 1, alpha=0.5, label='Intersected')
# census_tracts_legend = plt.Line2D([0], [0], color='black', lw=2, linestyle='-', label='Census Tracts')
# reporting_districts_legend = plt.Line2D([0], [0], color='red', lw=2, linestyle='-', label='Reporting Districts')

# # Add legend
# ax.legend(handles=[intersected_legend, census_tracts_legend, reporting_districts_legend], prop={'size': 50})

# # Turn off axis lines
# ax.axis('off')

# # # Save the plot
# # plt.savefig('CT_to_RD_merge.png')

# # # Show the plot
# plt.show()


In [7]:
# Make it a normal dataframe or else a .csv won't work
no_geo = intersected.drop(columns={'geometry'})

In [8]:
# Save the resulting data to a new file
no_geo.to_csv('output/CT_to_RD_lookup.csv')

# Merge with community context variables

### Income

In [6]:
# Define the folder path
income_folder = 'data/ACSST5Y2017-2022.S1901'

# Initialize an empty DataFrame to hold the concatenated data
income = pd.DataFrame()

# Rename columns using a dictionary
income_rename_dict = {
    'Geography': 'Census_Tract_ID',
    'Estimate!!Households!!Median income (dollars)': 'Median_Household_Income_Dollars'
}

income_columns_to_keep = ['Census_Tract_ID', 'Median_Household_Income_Dollars']

# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(income_folder, '*.csv'))

# Loop through all CSV files in the folder
for file_path in csv_files:
    try:
        # Extract the year from the file name (last 4 digits before the extension)
        file_name = os.path.basename(file_path)
        year = file_name[-19:-15]
        
        # Read the file into a DataFrame
        data = pd.read_csv(file_path)
        
        # Reset the header
        data.columns = data.iloc[0]
        data = data[1:].reset_index(drop=True)
        
        # Rename columns
        data = data.rename(columns=income_rename_dict)
        
        # Drop everything except specific values
        data = data.loc[:, income_columns_to_keep]
        
        # Splice off the last 6 digits to match our lookup table
        data['Census_Tract_ID'] = data['Census_Tract_ID'].str[-6:]
        
        # Replace specified values so that we don't have strings
        data['Median_Household_Income_Dollars'] = data['Median_Household_Income_Dollars'].replace({'250,000+': 250000})
        
        # Drop where - or *
        data = data[(data['Median_Household_Income_Dollars'] != '-') & (data['Median_Household_Income_Dollars'] != '*')]
        data['Median_Household_Income_Dollars'] = data['Median_Household_Income_Dollars'].astype(int)
        
        # Add the year as a column
        data['Year'] = year
        
        # Concatenate the data to the main DataFrame
        income = pd.concat([income, data], ignore_index=True)
        
    except Exception as e:
        print(f"An error occurred with file {file_path}: {e}")


In [8]:
# Merge income to census dataset
merged_data = pd.merge(intersected, income, how='left')

# Drop where NA
merged_data.dropna(subset={"Median_Household_Income_Dollars"}, inplace=True)

In [9]:
# List of column names to convert to integers
columns_to_convert = ['Census_Tract_ID', 'Reporting_District_ID','Intersecting_Area_SqF','Median_Household_Income_Dollars']

# Convert columns to integers
merged_data[columns_to_convert] = merged_data[columns_to_convert].astype(int)

### Race

In [10]:
# Define the folder path
race_folder = 'data/ACSDT5Y2017-2022.B03002'

# Initialize an empty DataFrame to hold the concatenated data
race = pd.DataFrame()

# Define the main and backup dictionaries for column renaming
race_rename_dict = {
    'Geography': 'Census_Tract_ID',
    'Estimate!!Total:': 'Total_Population',
    'Estimate!!Total:!!Not Hispanic or Latino:!!White alone': 'Population_White',
    'Estimate!!Total:!!Not Hispanic or Latino:!!Black or African American alone': 'Population_Black_African_American',
    'Estimate!!Total:!!Not Hispanic or Latino:!!American Indian and Alaska Native alone': 'Population_American_Indian_Alaska_Native',
    'Estimate!!Total:!!Not Hispanic or Latino:!!Asian alone': 'Population_Asian',
    'Estimate!!Total:!!Not Hispanic or Latino:!!Native Hawaiian and Other Pacific Islander alone': 'Population_Native_Hawaiian_Other_Pacific_Islander',
    'Estimate!!Total:!!Not Hispanic or Latino:!!Some other race alone': 'Population_Some_Other_Race',
    'Estimate!!Total:!!Not Hispanic or Latino:!!Two or more races:': 'Population_Two_Or_More_Races',
    'Estimate!!Total:!!Hispanic or Latino:': 'Total_Population_Hispanic_Latino'
}

# Use for 2017 and 2018
race_rename_dict_backup = {
    'Geography': 'Census_Tract_ID',
    'Estimate!!Total': 'Total_Population',
    'Estimate!!Total!!Not Hispanic or Latino!!White alone': 'Population_White',
    'Estimate!!Total!!Not Hispanic or Latino!!Black or African American alone': 'Population_Black_African_American',
    'Estimate!!Total!!Not Hispanic or Latino!!American Indian and Alaska Native alone': 'Population_American_Indian_Alaska_Native',
    'Estimate!!Total!!Not Hispanic or Latino!!Asian alone': 'Population_Asian',
    'Estimate!!Total!!Not Hispanic or Latino!!Native Hawaiian and Other Pacific Islander alone': 'Population_Native_Hawaiian_Other_Pacific_Islander',
    'Estimate!!Total!!Not Hispanic or Latino!!Some other race alone': 'Population_Some_Other_Race',
    'Estimate!!Total!!Not Hispanic or Latino!!Two or more races': 'Population_Two_Or_More_Races',
    'Estimate!!Total!!Hispanic or Latino': 'Total_Population_Hispanic_Latino'
}


# List of columns to keep based on the rename dictionary
race_columns_to_keep = list(race_rename_dict.values())

# Population columns
population_columns = ['Total_Population', 'Population_White', 'Population_Black_African_American',
                      'Population_American_Indian_Alaska_Native', 'Population_Asian',
                      'Population_Native_Hawaiian_Other_Pacific_Islander', 'Population_Some_Other_Race',
                      'Population_Two_Or_More_Races', 'Total_Population_Hispanic_Latino']

# Get all CSV files in the folder
csv_files = glob.glob(os.path.join(race_folder, '*.csv'))
# Loop through all CSV files in the folder
for file_path in csv_files:
    try:
        # Extract the year from the file name (last 4 digits before the extension)
        file_name = os.path.basename(file_path)
        year = file_name[-20:-16]
        
        # Read the file into a DataFrame
        data = pd.read_csv(file_path, skiprows=1)
        
        # Check if the column names in the file match the expected column names
        if set(race_rename_dict.keys()).issubset(data.columns):
            rename_dict_to_use = race_rename_dict
        elif set(race_rename_dict_backup.keys()).issubset(data.columns):
            rename_dict_to_use = race_rename_dict_backup
        else:
            print(f"No suitable column renaming dictionary found for file {file_path}. Skipping...")
            continue

        # Rename columns
        data = data.rename(columns=rename_dict_to_use)

        # Drop columns not present in the rename dictionary
        data = data[race_columns_to_keep]

        # Splice off the last 6 digits to match our lookup table
        data['Census_Tract_ID'] = data['Census_Tract_ID'].str[-6:].astype(int)

        data[population_columns] = data[population_columns].apply(pd.to_numeric, errors='coerce')

        # Calculate percentage of each race
        data['Percent_White'] = (data['Population_White'] / data['Total_Population']) * 100
        data['Percent_Black_African_American'] = (data['Population_Black_African_American'] / data['Total_Population']) * 100
        data['Percent_American_Indian_Alaska_Native'] = (data['Population_American_Indian_Alaska_Native'] / data['Total_Population']) * 100
        data['Percent_Asian'] = (data['Population_Asian'] / data['Total_Population']) * 100
        data['Percent_Native_Hawaiian_Other_Pacific_Islander'] = (data['Population_Native_Hawaiian_Other_Pacific_Islander'] / data['Total_Population']) * 100
        data['Percent_Some_Other_Race'] = (data['Population_Some_Other_Race'] / data['Total_Population']) * 100
        data['Percent_Two_Or_More_Races'] = (data['Population_Two_Or_More_Races'] / data['Total_Population']) * 100
        data['Percent_Hispanic_Latino'] = (data['Total_Population_Hispanic_Latino'] / data['Total_Population']) * 100

        # Add the year as a column
        data['Year'] = year

        # Concatenate the data to the main DataFrame
        race = pd.concat([race, data], ignore_index=True)
    
    except Exception as e:
        print(f"An error occurred with file {file_path}: {e}")


In [11]:
# Merge income to census dataset
merged_data = pd.merge(merged_data, race, on=['Census_Tract_ID','Year'], how='left')

# Create community context file
Area-weighted average variables by reporting district

### Compute area-weighted averages and aggregate by reporting district

In [12]:
# List of variables to aggregate
variables = ['Median_Household_Income_Dollars','Percent_White', 'Percent_Black_African_American',
                      'Percent_American_Indian_Alaska_Native', 'Percent_Asian',
                      'Percent_Native_Hawaiian_Other_Pacific_Islander',
                      'Percent_Some_Other_Race', 'Percent_Two_Or_More_Races',
                      'Percent_Hispanic_Latino']

grouped = merged_data

# Calculate the product of each variable and area
for var in variables:
    grouped[f'{var}_Area_Product'] = grouped[var] * grouped['Intersecting_Area_SqF']

# Group by the reporting district ID for each variable
agg_dict = {f'{var}_Area_Product': 'sum' for var in variables}
agg_dict['Intersecting_Area_SqF'] = 'sum'
grouped = grouped.groupby(['Reporting_District_ID','Year']).agg(agg_dict)

# Calculate the area-weighted average for each variable
for var in variables:
    grouped[f'Weighted_Avg_{var}'] = grouped[f'{var}_Area_Product'] / grouped['Intersecting_Area_SqF']

# Round the weighted average calculations
grouped = grouped.round(decimals=2)

In [13]:
# Selecting only the weighted average columns
comm_context = grouped[[f'Weighted_Avg_{var}' for var in variables]]

# Resetting the index to make 'Reporting_District_ID' a regular column
comm_context.reset_index(inplace=True)

In [14]:
# Group merged_data by Reporting_District_ID and Year and aggregate the list of intersecting census tracts
census_tracts_list = merged_data.groupby(['Reporting_District_ID', 'Year'])['Census_Tract_ID'].apply(list).reset_index()

# Merge census_tracts_list with comm_context on Reporting_District_ID
comm_context = pd.merge(comm_context, census_tracts_list, on=['Reporting_District_ID', 'Year'])

# Reorder columns in comm_context DataFrame
comm_context = comm_context[['Reporting_District_ID', 'Year', 'Census_Tract_ID'] + [col for col in comm_context.columns if col not in ['Reporting_District_ID', 'Year', 'Census_Tract_ID']]]

In [80]:
comm_context.head()

,Reporting_District_ID,Year,Census_Tract_ID,Weighted_Avg_Median_Household_Income_Dollars,Weighted_Avg_Percent_White,Weighted_Avg_Percent_Black_African_American,Weighted_Avg_Percent_American_Indian_Alaska_Native,Weighted_Avg_Percent_Asian,Weighted_Avg_Percent_Native_Hawaiian_Other_Pacific_Islander,Weighted_Avg_Percent_Some_Other_Race,Weighted_Avg_Percent_Two_Or_More_Races,Weighted_Avg_Percent_Hispanic_Latino
0,101,2017,"[197300, 197600, 197700, 980010]",36777.61,15.60,4.56,1.31,42.52,0.17,0.00,2.72,33.11
1,101,2018,"[197300, 197600, 197700]",44768.04,17.25,2.93,1.75,43.69,0.00,0.00,3.50,30.88
2,101,2019,"[197300, 197600, 197700, 980010]",54756.93,22.76,3.00,2.21,42.68,0.00,0.00,4.06,25.29
3,101,2020,"[197300, 197600, 197700, 980010]",52272.88,22.31,2.04,1.92,44.07,0.00,0.00,4.94,24.72
4,101,2021,"[197300, 197600, 197700, 980010]",55570.89,20.46,1.83,1.94,41.67,0.00,0.79,3.65,29.67


### Export to CSV

In [16]:
comm_context.to_csv('output/community_context_variables.csv')

# Create LAPD data file (temporal)

### Load and clean files

#### Arrests 
- Data from 2017-01-01 to 2023-12-31

In [16]:
# Define the folder path
arrests_folder = 'data/LAPD_Arrests'

# Initialize an empty DataFrame to hold the concatenated data
arrests = pd.DataFrame()

# Loop through all files in the folder
for filename in os.listdir(arrests_folder):
    # Construct the full file path
    file_path = os.path.join(arrests_folder, filename)
    
    # Check if the path is a file (and not a directory)
    if os.path.isfile(file_path):
        # Read the file into a DataFrame (assuming CSV format for this example)
        data = pd.read_csv(file_path)
        
        # Concatenate the data to the main DataFrame
        arrests = pd.concat([arrests, data], ignore_index=True)

In [18]:
arrests_rename_dict = {"Arrest Date":"Date",
                       "Reporting District":"Reporting_District_ID"
                      }

arrests = arrests.rename(columns=arrests_rename_dict)


In [20]:
arrests['Date'] = pd.to_datetime(arrests['Date'], format='mixed').dt.date

In [21]:
grouped_arrests = arrests.groupby(['Date', 'Reporting_District_ID']).size().reset_index(name='Count')

In [22]:
grouped_arrests = grouped_arrests.rename(columns={"Count":"Arrest_Count"})

In [31]:
# Ensure 'Date' column is in datetime format
grouped_arrests['Date'] = pd.to_datetime(grouped_arrests['Date'])

# Convert start_date and end_date to Timestamp (datetime)
start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2023-12-31')

grouped_arrests = grouped_arrests[(grouped_arrests['Date'] >= start_date) & (grouped_arrests['Date'] <= end_date)]

#### Calls for service
- Data from 2017-01-01 to 2023-12-31

In [57]:
# Define the folder path
calls_for_service_folder = 'data/LAPD_CallsForService'

# Initialize an empty DataFrame to hold the concatenated data
calls = pd.DataFrame()

# Loop through all files in the folder
for filename in os.listdir(calls_for_service_folder):
    # Construct the full file path
    file_path = os.path.join(calls_for_service_folder, filename)
    
    # Check if the path is a file (and not a directory)
    if os.path.isfile(file_path):
        # Read the file into a DataFrame (assuming CSV format for this example)
        data = pd.read_csv(file_path)
        if file_path == "data/LAPD_CallsForService/LAPD_Calls_for_Service_2017_20240530.csv":
            cols_rename_dict = {"Incident Number": "Incident_Number",
                                "Reporting District": "Rpt_Dist",	
                                "Area Occurred": "Area_Occ",
                                "Dispatch Date": "Dispatch_Date",
                                "Dispatch Time": "Dispatch_Time",
                                "Call Type Code": "Call_Type_Code",
                                "Call Type Description":"Call_Type_Text"}
            data = data.rename(columns=cols_rename_dict)

            reorder = list(cols_rename_dict.values())
            
            # Reorder the DataFrame columns
            data = data[reorder]
            
        # Concatenate the data to the main DataFrame
        calls = pd.concat([calls, data], ignore_index=True)

In [58]:
calls_rename_dict = {"Rpt_Dist":"Reporting_District_ID","Dispatch_Date":"Date"}

calls = calls.rename(columns=calls_rename_dict)

In [59]:
calls = calls.dropna(subset=['Reporting_District_ID'])

In [60]:
calls['Date'] = pd.to_datetime(calls['Date'], format='mixed').dt.date
calls['Date'] = pd.to_datetime(calls['Date'])
calls['Reporting_District_ID'] = calls['Reporting_District_ID'].astype(int)

In [61]:
grouped_calls = calls.groupby(['Date', 'Reporting_District_ID']).size().reset_index(name='Count')

In [62]:
grouped_calls = grouped_calls.rename(columns={"Count":"Call_Count"})

### Merge with each other

In [63]:
merged_lapd_data = pd.merge(grouped_arrests, grouped_calls, on=["Date","Reporting_District_ID"],how='left')

In [64]:
merged_lapd_data.sort_values(by=["Date","Reporting_District_ID"])

,Date,Reporting_District_ID,Arrest_Count,Call_Count
0,2017-01-01,111,1,15.0
1,2017-01-01,121,1,1.0
2,2017-01-01,134,1,1.0
3,2017-01-01,153,3,10.0
4,2017-01-01,156,1,4.0
...,...,...,...,...
383514,2023-12-31,2102,2,3.0
383515,2023-12-31,2148,1,1.0
383516,2023-12-31,2156,1,4.0
383517,2023-12-31,2159,1,1.0


### Export to CSV

In [65]:
merged_lapd_data.to_csv('output/lapd_demand_variables.csv')

### Merge with reporting district

In [61]:
# Modify RD to match other files
reporting_districts['Reporting_District_ID'] = reporting_districts['RD'].astype(str).str.lstrip('0').astype(int)

In [62]:
merged_lapd_geo = pd.merge(merged_lapd_data, reporting_districts, on=['Reporting_District_ID'], how='left')

In [63]:
merged_lapd_geo.head()

,Date,Reporting_District_ID,Arrest_Count,Call_Count,OBJECTID,RD,S_TYPE,COMMTYPE,NAME,ST_NAME,OMEGA_LABE,LABEL,AGENCY,STATION,OMEGA_NAME,Shape_Leng,Shape_Area,geometry
0,2020-01-01,111,1,23.0,52.0,0111,LAPD,City,Los Angeles,Central Division,LAPD 0111,LAPD Central Division,LAPD,CENTRAL,LAPD CENTRAL 0111,14217.874862,1.074055e+07,"POLYGON ((385956.840 3770249.393, 385949.617 3..."
1,2020-01-01,112,1,3.0,53.0,0112,LAPD,City,Los Angeles,Central Division,LAPD 0112,LAPD Central Division,LAPD,CENTRAL,LAPD CENTRAL 0112,7869.489268,1.797777e+06,"POLYGON ((385618.165 3768954.024, 385612.070 3..."
2,2020-01-01,119,1,3.0,55.0,0119,LAPD,City,Los Angeles,Central Division,LAPD 0119,LAPD Central Division,LAPD,CENTRAL,LAPD CENTRAL 0119,12149.891089,9.386648e+06,"POLYGON ((386610.402 3768718.375, 386554.222 3..."
3,2020-01-01,122,1,NaN,59.0,0122,LAPD,City,Los Angeles,Central Division,LAPD 0122,LAPD Central Division,LAPD,CENTRAL,LAPD CENTRAL 0122,6115.130700,2.213659e+06,"POLYGON ((385144.936 3769118.819, 385136.139 3..."
4,2020-01-01,127,2,NaN,62.0,0127,LAPD,City,Los Angeles,Central Division,LAPD 0127,LAPD Central Division,LAPD,CENTRAL,LAPD CENTRAL 0127,5423.912294,1.189909e+06,"POLYGON ((385758.346 3768853.570, 385757.102 3..."
